# Autoresearch Experiment Analysis

Analysis of autonomous hyperparameter tuning results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV with the wider-search schema.
df = pd.read_csv("results.tsv", sep="\t")

numeric_cols = ["val_bpb", "delta_vs_parent", "delta_vs_best", "memory_gb"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["status"] = df["status"].str.strip().str.upper()
df["tags"] = df.get("tags", "").fillna("")
df["description"] = df["description"].fillna("")
df["experiment"] = np.arange(len(df))

valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)
valid["experiment"] = np.arange(len(valid))
valid["running_best"] = valid["val_bpb"].cummin()
valid["is_new_best"] = valid["val_bpb"] == valid["running_best"]

baseline_bpb = valid.loc[0, "val_bpb"] if len(valid) else np.nan
best_row = valid.loc[valid["val_bpb"].idxmin()] if len(valid) else None

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_candidate = counts.get("CANDIDATE", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_candidate + n_discard
if n_decided > 0:
    print(f"\nRetention rate: {(n_keep + n_candidate)}/{n_decided} = {(n_keep + n_candidate) / n_decided:.1%}")
if len(valid) > 0:
    print(f"New global bests found: {int(valid['is_new_best'].sum())}")

In [ ]:
# Show all retained experiments: committed improvements and frontier candidates.
retained = df[df["status"].isin(["KEEP", "CANDIDATE"])].copy()
print(f"Retained experiments ({len(retained)} total):\n")
for i, row in retained.iterrows():
    bpb = row["val_bpb"]
    desc = row["description"]
    tags = row["tags"]
    print(f"  #{i:3d}  {row['status']:<9}  bpb={bpb:.6f}  mem={row['memory_gb']:.1f}GB  {desc}  [{tags}]")

## Val BPB Over Time

Track how the best val_bpb evolves as experiments progress under the wider search. The step line shows the running global best, while retained side branches remain visible as `KEEP` and `CANDIDATE` points.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Focus on the region near the baseline, as in the original plot.
plot_df = valid[valid["val_bpb"] <= baseline_bpb + 0.0005].copy()

disc = plot_df[plot_df["status"] == "DISCARD"]
cand = plot_df[plot_df["status"] == "CANDIDATE"]
keep = plot_df[plot_df["status"] == "KEEP"]
new_best = plot_df[plot_df["is_new_best"]]

ax.scatter(disc["experiment"], disc["val_bpb"],
           c="#cccccc", s=12, alpha=0.45, zorder=1, label="Discarded")
ax.scatter(cand["experiment"], cand["val_bpb"],
           c="#f39c12", s=32, alpha=0.8, zorder=3, label="Candidate", edgecolors="black", linewidths=0.4)
ax.scatter(keep["experiment"], keep["val_bpb"],
           c="#2ecc71", s=46, alpha=0.9, zorder=4, label="Keep", edgecolors="black", linewidths=0.5)

ax.step(valid["experiment"], valid["running_best"], where="post", color="#1f7a8c",
        linewidth=2.2, alpha=0.9, zorder=2, label="Running global best")

ax.scatter(new_best["experiment"], new_best["val_bpb"],
           c="#0b3954", s=60, marker="D", zorder=5, label="New global best", edgecolors="white", linewidths=0.6)

for _, row in new_best.iterrows():
    desc = str(row["description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (row["experiment"], row["val_bpb"]),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#0b3954", alpha=0.95,
                rotation=25, ha="left", va="bottom")

n_total = len(df)
n_new_best = int(valid["is_new_best"].sum())
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation BPB (lower is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_new_best} Global Best Updates", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

best_bpb = valid["val_bpb"].min()
margin = max((baseline_bpb - best_bpb) * 0.15, 0.0001)
ax.set_ylim(best_bpb - margin, baseline_bpb + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
new_best = valid[valid["is_new_best"]].copy()
best_bpb = valid["val_bpb"].min()

print(f"Baseline val_bpb:  {baseline_bpb:.6f}")
print(f"Best val_bpb:      {best_bpb:.6f}")
print(f"Total improvement: {baseline_bpb - best_bpb:.6f} ({(baseline_bpb - best_bpb) / baseline_bpb * 100:.2f}%)")
print(f"Best experiment:   {best_row['description']}")
print(f"Best commit:       {best_row['commit']}")
print()

print("Cumulative effort per new global best:")
for _, row in new_best.iterrows():
    desc = str(row["description"]).strip()
    print(f"  Experiment #{int(row['experiment']):3d}: bpb={row['val_bpb']:.6f}  {desc}")

## Top Hits (New Global Best Updates)

In [ ]:
# Measure each new-best update against the previous global best.
hits = valid[valid["is_new_best"]].copy()
hits["prev_best_bpb"] = hits["val_bpb"].shift(1)
hits["delta"] = hits["prev_best_bpb"] - hits["val_bpb"]
hits = hits.iloc[1:].copy()
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'BPB':>10}  Description")
print("-" * 90)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_bpb']:.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")